# Time-bound Runs with Bedrock OpenAI-compatible API

This notebook demonstrates how to use LLMeter's **time-bound run** feature to measure
sustained throughput and latency over a fixed time window, using Amazon Bedrock's
[OpenAI-compatible Chat Completion API](https://docs.aws.amazon.com/bedrock/latest/userguide/bedrock-openai-chat.html).

Instead of specifying a fixed number of requests, you set a `run_duration` in seconds
and LLMeter sends requests continuously until the time expires — giving you a realistic
picture of steady-state performance.

We also cover:
- **Live progress-bar statistics** (rpm, latency percentiles, tokens/s)
- **Low-memory mode** for large-scale runs
- **Custom progress-bar stats** configuration

## Setup

Install LLMeter with plotting extras, the OpenAI SDK, and the Bedrock token generator.

In [1]:
# %pip install "llmeter[plotting]<1" openai aws-bedrock-token-generator

In [2]:
import os

from llmeter.endpoints.openai import OpenAICompletionStreamEndpoint
from llmeter.runner import Runner

## Configure the Bedrock OpenAI-compatible endpoint

Amazon Bedrock exposes an [OpenAI-compatible Chat Completions API](https://docs.aws.amazon.com/bedrock/latest/userguide/bedrock-openai-chat.html)
accessible via the `bedrock-mantle` endpoint. Authentication uses a temporary token
generated from your AWS credentials via `aws-bedrock-token-generator`.

We use `OpenAICompletionStreamEndpoint` from LLMeter, which works with any
OpenAI Chat Completions-compatible API — including Bedrock's mantle endpoint.

In [3]:
from aws_bedrock_token_generator import provide_token

AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

# OpenAI GPT-OSS models available on the mantle endpoint:
#   openai.gpt-oss-120b  — larger model for complex tasks (120B parameters)
#   openai.gpt-oss-20b   — smaller, faster model (20B parameters)
# Use the Models API to discover all available models in your region.
MODEL_ID = "openai.gpt-oss-120b"  # Choose a model available via the Chat Completions API
BASE_URL = f"https://bedrock-mantle.{AWS_REGION}.api.aws/v1"

# Generate temporary token for Bedrock authentication
token = provide_token(region=AWS_REGION)

bedrock_endpoint = OpenAICompletionStreamEndpoint(
    model_id=MODEL_ID,
    endpoint_name="bedrock-mantle",
    provider="bedrock",
    base_url=BASE_URL,
    api_key=token,
)

print(f"Region: {AWS_REGION}")
print(f"Model: {MODEL_ID}")
print(f"Endpoint: {BASE_URL}")

Region: us-west-2
Model: openai.gpt-oss-120b
Endpoint: https://bedrock-mantle.us-west-2.api.aws/v1


### Verify the endpoint

Send a single request to confirm the endpoint is working and LLMeter captures
the expected metrics.

In [4]:
sample_payload = OpenAICompletionStreamEndpoint.create_payload(
    "Explain the difference between latency and throughput in 3 sentences.",
    max_tokens=256,
)

response = bedrock_endpoint.invoke(sample_payload)
print(response)

{
    "response_text": "Latency is the time it takes for a single piece of data to travel from source to destination, i.e., the delay before a response begins. Throughput measures how much data can be transferred successfully over a network or system in a given period of time, typically expressed in bits per second or requests per second. In short, latency is about *speed of response* for one item, while throughput is about *volume of work* that can be handled overall.",
    "input_payload": {
        "messages": [
            {
                "role": "user",
                "content": "Explain the difference between latency and throughput in 3 sentences."
            }
        ],
        "max_tokens": 256,
        "model": "openai.gpt-oss-120b",
        "stream": true,
        "stream_options": {
            "include_usage": true
        }
    },
    "id": "chatcmpl-e33a8b37-ad37-4946-9f05-8ce91013245f",
    "input_prompt": "Explain the difference between latency and throughput in 3 

You should see `time_to_first_token`, `time_to_last_token`, and token counts in the
response. If you see an error, check your AWS credentials and model access.

## Count-bound run (baseline)

First, let's run a traditional count-bound test for comparison.

In [5]:
runner = Runner(
    bedrock_endpoint,
    output_path=f"outputs/{MODEL_ID}",
)

count_result = await runner.run(
    payload=sample_payload,
    n_requests=5,
    clients=3,
    run_name="count-bound-baseline",
)

print(f"Total requests: {count_result.total_requests}")
print(f"Duration: {count_result.total_test_time:.1f}s")
print(f"RPM: {count_result.stats['requests_per_minute']:.1f}")

Total requests:   0%|          | 0/15 [00:00<?, ?it/s]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 99.6,p50_ttft 1.118s,p50_ttlt 2.205s,input_tokens 1155,fail 0
output_tps 215.5 tok/s,p90_ttft 3.083s,p90_ttlt 5.363s,output_tokens 1947,
p50_tps 182.0 tok/s,,,,


Total requests: 15
Duration: 14.3s
RPM: 63.1


## Time-bound run

Now let's run the same endpoint for a fixed duration instead. This is useful when you
want to measure **sustained throughput** — how many requests the endpoint can handle
over a realistic time window.

Set `run_duration` (in seconds) instead of `n_requests`. Each client sends requests
continuously until the duration expires.

In [6]:
duration_result = await runner.run(
    payload=sample_payload,
    run_duration=30,  # Run for 30 seconds
    clients=3,
    run_name="time-bound-30s",
)

print(f"Total requests completed: {duration_result.total_requests}")
print(f"Actual duration: {duration_result.total_test_time:.1f}s")
print(f"RPM: {duration_result.stats['requests_per_minute']:.1f}")
print(f"p50 TTFT: {duration_result.stats['time_to_first_token-p50']:.3f}s")
print(f"p90 TTLT: {duration_result.stats['time_to_last_token-p90']:.3f}s")

Elapsed:           | 0/30s [00:00]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 58.0,p50_ttft 1.180s,p50_ttlt 2.955s,input_tokens 2233,fail 0
output_tps 125.0 tok/s,p90_ttft 3.765s,p90_ttlt 6.345s,output_tokens 3750,
p50_tps 86.3 tok/s,,,,


Total requests completed: 29
Actual duration: 32.4s
RPM: 53.7
p50 TTFT: 1.180s
p90 TTLT: 6.345s


Notice that during the run, you see two progress bars:
- **Elapsed**: a time bar filling up as seconds pass
- **Requests**: a counter with live stats (rpm, latency percentiles, tokens/s, etc.)

## Load test: scaling with more clients

Time-bound runs are great for exploring how throughput scales with concurrency.
LLMeter's `LoadTest` experiment automates this — it runs each concurrency level
for the same duration and collects results for comparison.

The `run_duration` parameter makes each concurrency level run for a fixed time
window, giving a fair comparison of sustained throughput.

In [7]:
from llmeter.experiments import LoadTest

load_test = LoadTest(
    endpoint=bedrock_endpoint,
    payload=sample_payload,
    sequence_of_clients=[1, 3, 5, 10],
    run_duration=30,  # Each concurrency level runs for 30 seconds
    output_path=f"outputs/{MODEL_ID}",
    test_name="time-bound-load-test",
)

load_test_result = await load_test.run()

Configurations:   0%|          | 0/4 [00:00<?, ?it/s]

Elapsed:           | 0/30s [00:00]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 26.8,p50_ttft 1.071s,p50_ttlt 1.927s,input_tokens 1001,fail 0
output_tps 59.8 tok/s,p90_ttft 3.160s,p90_ttlt 6.137s,output_tokens 1741,
p50_tps 184.6 tok/s,,,,


Elapsed:           | 0/30s [00:00]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 75.0,p50_ttft 0.886s,p50_ttlt 2.035s,input_tokens 2849,fail 0
output_tps 171.7 tok/s,p90_ttft 2.327s,p90_ttlt 4.703s,output_tokens 5079,
p50_tps 113.6 tok/s,,,,


Elapsed:           | 0/30s [00:00]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 126.6,p50_ttft 1.181s,p50_ttlt 2.388s,input_tokens 4851,fail 0
output_tps 281.6 tok/s,p90_ttft 2.326s,p90_ttlt 4.191s,output_tokens 8411,
p50_tps 114.9 tok/s,,,,


Elapsed:           | 0/30s [00:00]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 214.4,p50_ttft 1.417s,p50_ttlt 3.034s,input_tokens 8162,fail 0
output_tps 481.3 tok/s,p90_ttft 3.600s,p90_ttlt 4.954s,output_tokens 14279,
p50_tps 103.0 tok/s,,,,


In [8]:
# Print summary for each concurrency level
for n_clients, result in sorted(load_test_result.results.items()):
    print(
        f"  {n_clients} clients: "
        f"{result.total_requests} requests, "
        f"{result.stats['requests_per_minute']:.0f} rpm, "
        f"p50 TTFT={result.stats.get('time_to_first_token-p50', 'N/A')}s"
    )

  1 clients: 13 requests, 23 rpm, p50 TTFT=1.0713617910005269s
  3 clients: 37 requests, 68 rpm, p50 TTFT=0.8862732499983395s
  5 clients: 63 requests, 118 rpm, p50 TTFT=1.181208833004348s
  10 clients: 106 requests, 196 rpm, p50 TTFT=1.4166151039971737s


## Custom progress-bar statistics

You can control which live stats appear on the progress bar via `progress_bar_stats`.
Each entry maps a display label to a field spec.

In [9]:
result = await runner.run(
    payload=sample_payload,
    run_duration=15,
    clients=3,
    run_name="custom-stats",
    progress_bar_stats={
        "rpm": "rpm",
        "p99_ttlt": ("time_to_last_token", "p99"),
        "tps": ("time_per_output_token", "p50", "inv"),
        "fail": "failed",
    },
)

Elapsed:           | 0/15s [00:00]

Throughput,TTLT,Errors
rpm 71.4,p99_ttlt 5.521s,fail 0
tps 75.3 tok/s,,


## Low-memory mode for large-scale runs

For long-running tests that generate thousands of responses, use `low_memory=True`
to avoid keeping all responses in memory. Responses are streamed to disk and stats
are computed incrementally.

This requires `output_path` to be set.

In [10]:
large_result = await runner.run(
    payload=sample_payload,
    run_duration=60,
    clients=10,
    run_name="large-low-memory",
    low_memory=True,
)

print(f"Total requests: {large_result.total_requests}")
print(f"RPM: {large_result.stats['requests_per_minute']:.1f}")
print(f"Responses in memory: {len(large_result.responses)}")
print(f"\nStats are available without loading responses:")
print(f"  p50 TTFT: {large_result.stats['time_to_first_token-p50']:.3f}s")
print(f"  p90 TTLT: {large_result.stats['time_to_last_token-p90']:.3f}s")

Elapsed:           | 0/60s [00:00]

Throughput,TTFT,TTLT,Tokens,Errors
rpm 220.1,p50_ttft 0.880s,p50_ttlt 2.555s,input_tokens 16940,fail 0
output_tps 485.8 tok/s,p90_ttft 3.128s,p90_ttlt 4.802s,output_tokens 29135,
p50_tps 102.0 tok/s,,,,


Total requests: 220
RPM: 209.5
Responses in memory: 0

Stats are available without loading responses:
  p50 TTFT: 0.880s
  p90 TTLT: 4.802s


In [11]:
# Load responses from disk when needed:
responses = large_result.load_responses()
print(f"Loaded {len(responses)} responses from disk")

Loaded 1045 responses from disk


## Comparing results

The `LoadTestResult` has a built-in `plot_results()` method that generates
standard charts (TTFT, TTLT, RPM, error rate, token throughput vs clients).

In [12]:
figs = load_test_result.plot_results(show=True)

### Custom charts: TPS and RPM vs clients

We can also build custom charts from the results. Here we plot the median output
tokens per second (TPS) and requests per minute (RPM) as a function of concurrency.

In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

clients_sorted = sorted(load_test_result.results.keys())
rpms = [load_test_result.results[c].stats["requests_per_minute"] for c in clients_sorted]

# Compute median TPS (1 / p50 time_per_output_token) for each concurrency level
tps_values = []
for c in clients_sorted:
    tpot_p50 = load_test_result.results[c].stats.get("time_per_output_token-p50")
    tps_values.append(1.0 / tpot_p50 if tpot_p50 and tpot_p50 > 0 else None)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Requests per Minute vs Clients", "Median Output Tokens/s vs Clients"),
)

fig.add_trace(
    go.Scatter(x=clients_sorted, y=rpms, mode="lines+markers", name="RPM"),
    row=1, col=1,
)
fig.add_trace(
    go.Scatter(x=clients_sorted, y=tps_values, mode="lines+markers", name="TPS (p50)"),
    row=1, col=2,
)

fig.update_xaxes(title_text="Clients", type="log")
fig.update_yaxes(title_text="RPM", row=1, col=1)
fig.update_yaxes(title_text="Tokens/s", row=1, col=2)
fig.update_layout(height=400, showlegend=False)
fig

In [14]:
from llmeter.plotting import boxplot_by_dimension

fig = go.Figure(layout=dict(title="Time to First Token by Client Count"))
for n_clients in clients_sorted:
    fig.add_trace(
        boxplot_by_dimension(
            load_test_result.results[n_clients],
            dimension="time_to_first_token",
            name=f"{n_clients} clients",
        )
    )
fig.update_xaxes(type="log", title="Time to First Token (s)")
fig

In [15]:
fig = go.Figure(layout=dict(title="Time to Last Token by Client Count"))
for n_clients in clients_sorted:
    fig.add_trace(
        boxplot_by_dimension(
            load_test_result.results[n_clients],
            dimension="time_to_last_token",
            name=f"{n_clients} clients",
        )
    )
fig.update_xaxes(type="log", title="Time to Last Token (s)")
fig

## Summary

| Feature | Parameter | Description |
|---|---|---|
| Time-bound runs | `run_duration=60` | Run for a fixed number of seconds instead of a fixed request count |
| Count-bound runs | `n_requests=100` | Traditional mode — fixed number of requests per client |
| Live stats | `progress_bar_stats={...}` | Customize which metrics appear on the progress bar |
| Low-memory mode | `low_memory=True` | Stream responses to disk, compute stats incrementally |